# Stage C / NB 13 — Agent output registry

Protocol reference: Section 9 Stage C NB 13. Feeds NB 14 (fusion) and NB 15 (reasoner).

## What this notebook is for

One tidy table, one row per `(image, agent, arm)`, carrying every agent's prediction,
continuous score, uncertainty and provenance. NB 14 and NB 15 read this instead of each
re-parsing seven differently-shaped Stage B outputs.

## The leakage boundary this notebook exists to enforce

Experiment E3 supplies the reasoner with **per-agent reliability metrics** — "A6's mRALE MAE is
3.9, A5's is 8.1, trust them accordingly". If those reliabilities were computed on the same
held-out images the reasoner is then scored on, the framework would be reading its own answer
key, and every E3 number would be worthless.

So reliability is computed **only on the inner-validation split** — the same 10% of training
groups Stage B used for checkpoint selection, regenerated here with the identical rule and seed.
Two artifacts are written and they are kept physically separate:

| file | contents | may the reasoner see it? |
| --- | --- | --- |
| `agent_registry.parquet` | per-image predictions, out-of-fold | yes, that is its input |
| `agent_reliability_inner.csv` | per-agent aggregate reliability | yes — computed on inner validation only |
| — | any statistic derived from test-fold rows | **never** |

The gate asserts, in code, that no test-fold row contributed to any reliability statistic. That
assertion is the deliverable of this notebook as much as the table is.

## A note on which agents exist

Stage B is partially complete. Agents whose prediction file is missing are recorded with a
reason rather than silently omitted — an agent absent from the registry is otherwise
indistinguishable from an agent that contributed nothing, and E1's leave-one-out would
misattribute the difference.

## Outputs (under `stage_C/nb13_registry/`)
`agent_registry.parquet` (and `.csv` fallback), `agent_reliability_inner.csv`,
`agent_availability.csv`, `evidence_index.jsonl`, `registry_manifest.json`, `gate_nb13.json`.

## 1. Imports and the Stage A path contract

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions live with the Stage B notebooks; every arm in Table 2 and every
# fusion/reasoner arm here must be scored by identical code or the comparison is invalid.
_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_B",
           Path.cwd().parent.parent / "notebooks" / "stage_B"]
for _candidate in _SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(f"cxr_metrics.py not found. Searched: {_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_C_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # identical to Stage B, so inner splits match exactly

print("Stage C output:", STAGE_C_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Configuration and Stage B source map

In [ ]:
NB13_DIR = STAGE_C_DIR / "nb13_registry"
NB13_DIR.mkdir(parents=True, exist_ok=True)

# An agent must cover at least this fraction of the internal cohort to enter the registry as
# usable. Below it, its rows are kept for auditing but it is flagged, because a sparse agent
# silently shrinks the intersection every fusion arm is fitted on.
MIN_AGENT_COVERAGE = 0.90

# When an agent produced several arms (A5 has three, A1/E4 has up to seven), only one enters
# the roster. Selection is by inner-validation MAE, never by test performance.
SELECT_BEST_ARM_PER_AGENT = True

# ---- Where each agent's out-of-fold predictions live --------------------------------------
# arm -> (agent id, directory, filename). Missing sources are skipped with a recorded reason
# rather than silently omitted, because an agent absent from the registry cannot be
# distinguished from an agent that contributed nothing.
AGENT_SOURCES = OrderedDict([
    ("A2_medgemma_lora",  ("A2", STAGE_B_DIR / "nb09_medgemma_lora",
                           "predictions_medgemma_lora.jsonl")),
    ("A3_qwen_lora",      ("A3", STAGE_B_DIR / "nb10_qwen_lora",
                           "predictions_qwen_lora.jsonl")),
    ("A4_nvreason",       ("A4", STAGE_B_DIR / "nb11_nvreason",
                           "predictions_nvreason.jsonl")),
    ("A5_cxformer",       ("A5", STAGE_B_DIR / "nb05_frozen_encoder",
                           "predictions_frozen.jsonl")),
    ("A6_biomedclip",     ("A6", STAGE_B_DIR / "nb08_biomedclip_entity",
                           "predictions_entity_probe.jsonl")),
    ("A1_anatomy",        ("A1", STAGE_B_DIR / "nb12_anatomy_aware",
                           "anatomy_aware_predictions.jsonl")),
    ("E0g_conventional",  ("E0g", STAGE_B_DIR / "nb06_conventional",
                           "predictions_conventional.jsonl")),
])

# Evidence channels the reasoner can read, beyond numeric predictions.
EVIDENCE_SOURCES = {
    "A6_entity_findings": STAGE_B_DIR / "nb08_biomedclip_entity" / "entity_findings.jsonl",
    "A4_nvreason_findings": STAGE_B_DIR / "nb11_nvreason" / "nvreason_findings.jsonl",
}


def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level.")
    frame = pd.read_csv(path)
    frame["image_key"] = "MIDRC::" + frame["filename"].astype(str)
    frame["mrale_right"] = frame["extent_right_numerical"] * frame["density_right_numerical"]
    frame["mrale_left"] = frame["extent_left_numerical"] * frame["density_left_numerical"]
    return frame


def inner_split_groups(folds, fold, fraction=INTERNAL_VALIDATION_FRACTION, seed=SEED):
    """
    Reproduce Stage B's inner validation split EXACTLY.

    Reliability statistics must be computed on data the agents were selected on, never on the
    held-out fold. Regenerating the split here with the same rule and seed is what makes the
    E3 prompts provably free of test-derived information.
    """
    pool = folds[folds["fold"] != fold].copy()
    # Stage B NB 09 groups by the image-directory path. NB 01's group_id can merge
    # cross-study near duplicates, so using it here would not reproduce Stage B exactly.
    pool["inner_group"] = pool["image_path"].map(lambda p: str(Path(p).parent))
    groups = sorted(pool["inner_group"].astype(str).unique())
    rng = random.Random(seed + fold)
    rng.shuffle(groups)
    n_validation = max(1, round(len(groups) * fraction))
    return set(groups[:n_validation])

## 3. Load folds and rebuild the inner split

The inner split is regenerated rather than read from disk, using Stage B's rule and seed. If
Stage B ever changes its splitting rule, this reproduction breaks loudly instead of silently
computing reliability on the wrong rows.

In [ ]:
folds = load_folds()
print(f"Internal cohort: {len(folds):,} images, {folds['group_id'].nunique():,} groups")

inner_validation_keys, inner_by_fold = set(), {}
for fold in range(N_FOLDS):
    groups = inner_split_groups(folds, fold)
    pool = folds[folds["fold"] != fold].copy()
    pool["inner_group"] = pool["image_path"].map(lambda p: str(Path(p).parent))
    keys = set(pool[pool["inner_group"].astype(str).isin(groups)]["image_key"].astype(str))
    inner_by_fold[fold] = keys
    inner_validation_keys |= keys
    print(f"  fold {fold}: inner-validation groups={len(groups):<5} images={len(keys):,}")

test_keys_by_fold = {fold: set(folds[folds["fold"] == fold]["image_key"].astype(str))
                     for fold in range(N_FOLDS)}
all_test_keys = set().union(*test_keys_by_fold.values())

# The property the whole notebook rests on: for a given fold, its inner-validation images are
# drawn only from the OTHER folds, so they can never be that fold's test images.
for fold in range(N_FOLDS):
    overlap = inner_by_fold[fold] & test_keys_by_fold[fold]
    if overlap:
        raise RuntimeError(
            f"fold {fold}: {len(overlap)} inner-validation images are also its TEST images. "
            "Reliability computed on these would leak the answer key into the E3 prompts.")
print()
print("Inner-validation / test disjointness verified for all five folds.")
print(f"Union of inner-validation images: {len(inner_validation_keys):,}")

## 4. Ingest every available Stage B agent

In [ ]:
truth_by_key = {}
for record in folds.to_dict("records"):
    truth_by_key[str(record["image_key"])] = {
        "gt_covid": record.get("covid_positive"),
        "gt_mrale_total": int(record["mrale_total_annotated"]),
        "gt_mrale_right": int(record["mrale_right"]),
        "gt_mrale_left": int(record["mrale_left"]),
        "severity_band": cm.severity_band(int(record["mrale_total_annotated"])),
        "fold": int(record["fold"]),
        "group_id": str(record["group_id"]),
    }

registry_rows, availability = [], []
for arm_label, (agent_id, directory, filename) in AGENT_SOURCES.items():
    path = directory / filename
    if not path.is_file():
        availability.append({"agent": agent_id, "source_arm": arm_label,
                             "path": str(path), "status": "MISSING",
                             "n_rows": 0, "n_images": 0, "coverage": 0.0,
                             "reason": "prediction file not found; notebook not yet run"})
        print(f"  [MISSING] {arm_label:<20} {path.name}")
        continue

    rows = cm.read_jsonl(path)
    internal = [r for r in rows if str(r.get("image_key", "")) in truth_by_key]
    by_arm = defaultdict(list)
    for row in internal:
        by_arm[row.get("arm", arm_label)].append(row)

    for arm, arm_rows in by_arm.items():
        keys = {str(r["image_key"]) for r in arm_rows}
        coverage = len(keys) / max(len(truth_by_key), 1)
        availability.append({
            "agent": agent_id, "source_arm": arm_label, "arm": arm, "path": str(path),
            "status": "OK" if coverage >= MIN_AGENT_COVERAGE else "SPARSE",
            "n_rows": len(arm_rows), "n_images": len(keys), "coverage": round(coverage, 4),
            "reason": "" if coverage >= MIN_AGENT_COVERAGE
                      else f"covers only {coverage:.1%} of the internal cohort",
        })
        for row in arm_rows:
            key = str(row["image_key"])
            truth = truth_by_key[key]
            registry_rows.append({
                "image_key": key, "agent": agent_id, "arm": arm,
                "fold": truth["fold"], "group_id": truth["group_id"],
                "mrale_total": row.get("mrale_total"),
                "mrale_right": row.get("mrale_right"),
                "mrale_left": row.get("mrale_left"),
                "mrale_total_expected": row.get("mrale_total_expected"),
                "mrale_uncertainty": row.get("mrale_uncertainty"),
                "covid_pred": row.get("covid_pred"),
                "covid_score": row.get("covid_score"),
                "valid": bool(row.get("valid", row.get("mrale_total") is not None)),
                "parse_error": row.get("parse_error"),
                "seconds": row.get("seconds"),
                **truth,
            })
        print(f"  [{'OK' if coverage >= MIN_AGENT_COVERAGE else 'SPARSE':<6}] "
              f"{agent_id}/{arm:<24} images={len(keys):<5} coverage={coverage:.3f}")

registry = pd.DataFrame(registry_rows)

# ---- Fold-relative inner-validation membership -------------------------------------------
# "Is this image inner validation?" has no fold-free answer. Fold k's inner-validation set is
# drawn from folds != k, so an image is inner validation FOR FOLD k and simultaneously TEST
# data for its own fold. A single `split` column collapses that distinction and, because
# inner_by_fold[k] never contains fold-k images, would label every row "test" -- leaving every
# downstream fit set empty. One indicator column per fold keeps the relation intact.
INNER_FOLD_COLUMNS = [f"inner_fold_{k}" for k in range(N_FOLDS)]
if len(registry):
    for k in range(N_FOLDS):
        registry[f"inner_fold_{k}"] = registry["image_key"].isin(inner_by_fold[k]).astype(int)
    registry["n_inner_memberships"] = registry[INNER_FOLD_COLUMNS].sum(axis=1)
    registry["split"] = np.where(registry["n_inner_memberships"] > 0,
                                 "inner_pool", "test_only")
    print("Fold-relative membership (rows usable for fitting/reliability per fold):")
    for k in range(N_FOLDS):
        n = int(registry[f"inner_fold_{k}"].sum())
        print(f"  inner_fold_{k}: {n:,} rows"
              + ("  <-- EMPTY, downstream fits would have no data" if n == 0 else ""))

pd.DataFrame(availability).to_csv(NB13_DIR / "agent_availability.csv", index=False)
print()
print(f"Registry rows: {len(registry):,}  agents={sorted(registry['agent'].unique()) if len(registry) else []}")
missing = [a["source_arm"] for a in availability if a["status"] == "MISSING"]
if missing:
    print(f"NOT YET AVAILABLE: {missing}")
    print("  Recorded rather than omitted: an agent absent from the registry cannot otherwise")
    print("  be distinguished from one that contributed nothing, and E1's leave-one-out would")
    print("  misattribute the difference.")

## 5. Reliability — inner validation only

Each agent's reliability is what E3 hands the reasoner. It is computed on the inner-validation
rows **of the fold the agent was selected on**, so it describes the agent's expected behaviour
without ever touching the images the framework is scored on.

**Reliability is per fold, and it has to be.** "Inner validation" is not a property of an
image; it is a relation between an image and a fold. Fold *k*'s inner-validation set is drawn
from folds != *k*, so those same images are test data for whichever fold actually held them
out. A single `split` column cannot express that — and worse, since `inner_by_fold[k]` never
contains fold-*k* images, a column defined as "is this image in its own fold's inner set" is
identically false, which would leave every downstream fit set empty and every fold silently
skipped.

Section 4 therefore emits one indicator column per fold. Row `scope = inner_validation_for_fold`
with `for_fold = k` is the row E3 hands to a reasoner scoring fold *k*. The pooled row exists
only for display and for choosing between an agent's arms.

In [ ]:
def finite_in_range(value, low, high):
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return number if np.isfinite(number) and low <= number <= high else None


def reliability_for(rows, label):
    # Pandas may preserve failed generations as NaN/inf. Normalize every prediction
    # before the shared metric boundary so invalid outputs receive the protocol penalty
    # and never become NumPy's minimum-int sentinel inside QWK.
    mrale_rows = [{"gt_mrale_total": finite_in_range(r.get("gt_mrale_total"), 0, 24),
                   "mrale_total": finite_in_range(r.get("mrale_total"), 0, 24),
                   "gt_mrale_right": finite_in_range(r.get("gt_mrale_right"), 0, 12),
                   "mrale_right": finite_in_range(r.get("mrale_right"), 0, 12),
                   "gt_mrale_left": finite_in_range(r.get("gt_mrale_left"), 0, 12),
                   "mrale_left": finite_in_range(r.get("mrale_left"), 0, 12)}
                  for r in rows
                  if finite_in_range(r.get("gt_mrale_total"), 0, 24) is not None]
    covid_rows = [r for r in rows if r["gt_covid"] in {"Yes", "No"}]
    entry = {"n": len(rows)}
    if mrale_rows:
        m = cm.mrale_metrics(mrale_rows)
        entry.update({"mrale_mae": round(m.get("mae", float("nan")), 4),
                      "mrale_rmse": round(m.get("rmse", float("nan")), 4),
                      "mrale_qwk": round(m.get("qwk", float("nan")), 4),
                      "mrale_spearman": round(m.get("spearman_rho", float("nan")), 4),
                      "mrale_coverage": round(m.get("coverage", float("nan")), 4)})
    if covid_rows:
        c = cm.classification_metrics(
            [r["gt_covid"] for r in covid_rows], [r.get("covid_pred") for r in covid_rows],
            [r.get("covid_score") for r in covid_rows])
        entry.update({"covid_auroc": round(c.get("auroc", float("nan")), 4),
                      "covid_balanced_accuracy": round(c.get("balanced_accuracy", float("nan")), 4),
                      "covid_brier": round(c.get("brier", float("nan")), 4)})
        # Calibration slope: logistic outcome regression on logit(p). A linear fit of
        # truth directly on probability is not a calibration slope and has no 1.0 ideal.
        scored = [(1 if r["gt_covid"] == "Yes" else 0,
                   finite_in_range(r.get("covid_score"), 0, 1))
                  for r in covid_rows
                  if finite_in_range(r.get("covid_score"), 0, 1) is not None]
        if len(scored) > 30 and len({s[0] for s in scored}) == 2:
            y = np.array([s[0] for s in scored], dtype=float)
            probability = np.clip(np.array([s[1] for s in scored], dtype=float),
                                  1e-6, 1 - 1e-6)
            logit = np.log(probability / (1 - probability)).reshape(-1, 1)
            if logit.std() > 1e-9:
                from sklearn.linear_model import LogisticRegression
                try:
                    calibration = LogisticRegression(penalty=None, solver="lbfgs",
                                                     max_iter=2000).fit(logit, y)
                except (TypeError, ValueError):
                    calibration = LogisticRegression(penalty="none", solver="lbfgs",
                                                     max_iter=2000).fit(logit, y)
                entry["covid_calibration_slope"] = round(
                    float(calibration.coef_[0, 0]), 4)
    return entry


reliability_rows = []
if len(registry):
    inner_pool = registry[registry["n_inner_memberships"] > 0]
    print(f"Rows in the inner-validation pool: {len(inner_pool):,} of {len(registry):,} "
          f"({len(inner_pool) / max(len(registry), 1):.1%})")

    # PER FOLD: this is what E3 actually consumes. Fold k's numbers are computed only on rows
    # flagged inner_fold_k, none of which are fold-k test images.
    for k in range(N_FOLDS):
        subset = registry[registry[f"inner_fold_{k}"] == 1]
        for (agent, arm), group in subset.groupby(["agent", "arm"]):
            entry = reliability_for(group.to_dict("records"), f"{agent}/{arm}/for_fold{k}")
            reliability_rows.append({"agent": agent, "arm": arm, "for_fold": k,
                                     "scope": "inner_validation_for_fold", **entry})

    # POOLED: for descriptive display only. Never used for fold-specific arm selection or
    # handed to a reasoner, because the pool contains every fold's outer-test images.
    for (agent, arm), group in inner_pool.groupby(["agent", "arm"]):
        entry = reliability_for(group.to_dict("records"), f"{agent}/{arm}")
        reliability_rows.append({"agent": agent, "arm": arm, "scope": "inner_validation",
                                 **entry})

reliability = pd.DataFrame(reliability_rows)
reliability.to_csv(NB13_DIR / "agent_reliability_inner.csv", index=False)
if len(reliability):
    pooled = reliability[reliability["scope"] == "inner_validation"]
    print()
    print(pooled[["agent", "arm", "n", "mrale_mae", "mrale_qwk", "covid_auroc",
                  "covid_balanced_accuracy"]].to_string(index=False))
    print()
    print("These pooled numbers are descriptive only. E3b receives the fold-specific rows")
    print("above, so neither reliability nor arm selection uses its own outer-test fold.")

## 6. Select one arm per agent, and check the choice is fold-independent

Agents that produced several arms contribute one to the roster. Selection is by
inner-validation MAE — never by test performance, which would be selection on the outcome.

There is a subtlety worth naming rather than glossing. The pooled inner-validation set is the
union over folds, and every image in it is test data for exactly one fold. Selecting on the
pool therefore lets fold *k*'s test rows influence which arm is later applied to fold *k*.
The effect is second-order — the choice is between arms of the *same* agent — but it is real.

So the choice is made **five times, once per fold**, and the cell checks whether they agree. If
one arm wins in every fold, no fold's test data determined the outcome and a single global
choice is safe. If they disagree, that is reported as a finding: the arms are within noise of
each other, and the manuscript should say so instead of presenting a tuned selection.

In [ ]:
def choose_arm(group):
    """Pick one arm from a per-agent group of reliability rows. Returns (arm, criterion)."""
    usable = group.dropna(subset=["mrale_mae"])
    if len(usable):
        return usable.sort_values("mrale_mae").iloc[0]["arm"], "inner-validation mRALE MAE"
    # No mRALE signal (A4 cannot express one); fall back to AUROC where available.
    usable = group.dropna(subset=["covid_auroc"])
    if len(usable):
        return (usable.sort_values("covid_auroc", ascending=False).iloc[0]["arm"],
                "inner-validation AUROC")
    return group.iloc[0]["arm"], "only arm available"


selected_arms, per_fold_choice, unstable = {}, defaultdict(dict), []
if SELECT_BEST_ARM_PER_AGENT and len(reliability):
    per_fold = reliability[reliability["scope"] == "inner_validation_for_fold"]
    for k in range(N_FOLDS):
        subset = per_fold[per_fold["for_fold"] == k]
        for agent, group in subset.groupby("agent"):
            per_fold_choice[agent][k] = choose_arm(group)[0]

    pooled = reliability[reliability["scope"] == "inner_validation"].copy()
    for agent, group in pooled.groupby("agent"):
        votes = Counter(per_fold_choice.get(agent, {}).values())
        if votes:
            arm = votes.most_common(1)[0][0]
            criterion = f"per-fold inner MAE, {votes[arm]}/{sum(votes.values())} folds agree"
            if len(votes) > 1:
                unstable.append((agent, dict(votes)))
        else:
            arm, criterion = choose_arm(group)
        row = group[group["arm"] == arm]
        mae = float(row.iloc[0]["mrale_mae"]) if len(row) and pd.notna(
            row.iloc[0].get("mrale_mae")) else None
        selected_arms[agent] = {"arm": arm, "criterion": criterion, "inner_mae": mae,
                                "n_candidates": int(len(group)),
                                "per_fold": per_fold_choice.get(agent, {})}
        print(f"  {agent}: {arm:<26} by {criterion} (from {len(group)} candidate arm(s))")

if unstable:
    print()
    print("ARM SELECTION IS NOT FOLD-INDEPENDENT for: "
          + ", ".join(f"{a} {v}" for a, v in unstable))
    print("  Different folds prefer different arms. The registry therefore keeps the")
    print("  choice fold-specific; no pooled majority arm is applied to an outer fold.")
    print("  Report the spread rather than inventing a global winner.")
elif selected_arms:
    print()
    print("Every agent's arm won in all five folds independently, so no fold's test data")
    print("determined the selection applied to it.")

# Selection is a relation between an arm and the fold it will be applied to. A global
# majority choice can be influenced by fold k's test rows through another fold's inner
# set. Emit one selector per fold and a convenience selector for each row's own outer fold.
SELECTED_FOLD_COLUMNS = [f"selected_for_fold_{k}" for k in range(N_FOLDS)]
for k, column in enumerate(SELECTED_FOLD_COLUMNS):
    registry[column] = registry.apply(
        lambda r, fold=k: per_fold_choice.get(r["agent"], {}).get(
            fold, selected_arms.get(r["agent"], {}).get("arm")) == r["arm"], axis=1)
registry["selected_for_outer_fold"] = registry.apply(
    lambda r: bool(r[f"selected_for_fold_{int(r['fold'])}"]), axis=1)
# Backward-compatible name: safe only for rows evaluated on their own outer fold.
registry["selected"] = registry["selected_for_outer_fold"]
print()
print(f"Outer-fold-selected rows: {int(registry['selected'].sum()):,} of {len(registry):,}")

## 7. Evidence channels and the intersection every fusion arm shares

In [ ]:
evidence_rows = []
for label, path in EVIDENCE_SOURCES.items():
    if not path.is_file():
        print(f"  [MISSING] {label}: {path.name}")
        continue
    n = 0
    for row in cm.read_jsonl(path):
        key = str(row.get("image_key", ""))
        if key not in truth_by_key:
            continue
        evidence_rows.append({"image_key": key, "channel": label,
                              "findings": row.get("findings", []),
                              "abstained": row.get("abstained"),
                              "n_findings": row.get("n_findings",
                                                    row.get("n_findings_reported"))})
        n += 1
    print(f"  [OK] {label}: {n:,} images")
if evidence_rows:
    cm.write_jsonl(NB13_DIR / "evidence_index.jsonl", evidence_rows)

# The intersection matters: every fusion and reasoner arm is fitted on the images ALL
# participating agents scored. A sparse agent silently shrinks it for everyone.
selected = registry[registry["selected"]] if len(registry) else registry
if len(selected):
    per_agent_keys = {a: set(g["image_key"]) for a, g in selected.groupby("agent")}
    intersection = set.intersection(*per_agent_keys.values()) if per_agent_keys else set()
    union = set().union(*per_agent_keys.values()) if per_agent_keys else set()
    print()
    print(f"Images covered by EVERY selected agent : {len(intersection):,}")
    print(f"Images covered by at least one         : {len(union):,}")
    print(f"Full internal cohort                   : {len(truth_by_key):,}")
    for agent, keys in sorted(per_agent_keys.items()):
        lost = len(union - keys)
        print(f"    {agent}: {len(keys):,} images"
              + (f"  (excludes {lost:,} that others cover)" if lost else ""))
    if len(intersection) < 0.9 * len(truth_by_key):
        print()
        print("  The intersection is well below the full cohort. NB 14/15 must state which")
        print("  denominator they used; comparing a fusion arm fitted on the intersection")
        print("  against a single agent scored on the full cohort is not a fair contrast.")
else:
    intersection, union = set(), set()

## 8. Write the registry and gate

In [ ]:
if len(registry):
    try:
        registry.to_parquet(NB13_DIR / "agent_registry.parquet", index=False)
        written = "agent_registry.parquet"
    except Exception as exc:
        registry.to_csv(NB13_DIR / "agent_registry.csv", index=False)
        written = f"agent_registry.csv (parquet unavailable: {type(exc).__name__})"
    print("Wrote", written)

failures, warnings = [], []

# THE assertion this notebook exists for: no test-fold row may inform reliability.
if len(registry):
    contaminated, empty_folds = 0, []
    for fold in range(N_FOLDS):
        marked = registry[registry[f"inner_fold_{fold}"] == 1]
        if not len(marked):
            empty_folds.append(fold)
        # The property: nothing flagged as fold k's inner validation may be a fold-k image.
        contaminated += int((marked["fold"] == fold).sum())
    if contaminated:
        failures.append(
            f"{contaminated} rows are flagged as inner validation for a fold they are the TEST "
            "data of. Every E3 reliability number for that fold would be computed on the "
            "answer key.")
    else:
        print("LEAKAGE ASSERTION PASSED: for every fold k, no row flagged inner_fold_k "
              "belongs to fold k.")
    if empty_folds:
        failures.append(
            f"inner_fold_{empty_folds} contain no rows, so NB 14 and NB 15 would have nothing "
            "to fit weights or compute reliability on for those folds. Check that "
            "inner_split_groups() reproduces Stage B's split.")

    # Every (image, agent, target fold) can select at most one arm. More than one would
    # duplicate an agent in downstream fusion; zero is permitted only for sparse coverage.
    for fold, column in enumerate(SELECTED_FOLD_COLUMNS):
        counts = registry[registry[column]].groupby(["image_key", "agent"]).size()
        duplicated = counts[counts > 1]
        if len(duplicated):
            failures.append(f"fold {fold}: {len(duplicated)} image/agent pairs select "
                            "more than one arm. Fusion features would be duplicated.")

    # A reliability statistic must never be derivable from a test row: verify by recomputing
    # one agent's MAE from test rows and confirming it differs from the stored figure.
    if len(reliability):
        sample = reliability[reliability["scope"] == "inner_validation"].head(1)
        if len(sample):
            agent, arm = sample.iloc[0]["agent"], sample.iloc[0]["arm"]
            test_rows = registry[(registry["agent"] == agent) & (registry["arm"] == arm)
                                 & (registry["n_inner_memberships"] == 0)]
            if len(test_rows) > 30:
                test_mae = cm.mrale_metrics([
                    {"gt_mrale_total": r["gt_mrale_total"], "mrale_total": r["mrale_total"]}
                    for r in test_rows.to_dict("records")
                    if finite_in_range(r.get("gt_mrale_total"), 0, 24) is not None]).get("mae")
                stored = sample.iloc[0].get("mrale_mae")
                print(f"  cross-check {agent}/{arm}: inner MAE {stored} vs test MAE "
                      f"{None if test_mae is None else round(test_mae, 4)} — these should "
                      "differ; identical values would suggest the same rows were used twice.")
else:
    failures.append("Registry is empty: no Stage B prediction file was found.")

if unstable:
    warnings.append(
        "Arm selection differs across folds for " + ", ".join(a for a, _ in unstable)
        + ". Fold-specific selectors are used downstream; report the instability.")

for entry in availability:
    if entry["status"] == "MISSING":
        warnings.append(f"{entry['source_arm']}: {entry['reason']} ({entry['path']}). "
                        "E1 cannot leave out an agent that was never present.")
    elif entry["status"] == "SPARSE":
        warnings.append(f"{entry['agent']}/{entry.get('arm')}: {entry['reason']}. It shrinks "
                        "the intersection every fusion arm is fitted on.")

present = {a["agent"] for a in availability if a["status"] in {"OK", "SPARSE"}}
required_for_e1 = {"A1", "A2", "A3", "A4", "A5", "A6"}
absent = sorted(required_for_e1 - present)
if absent:
    warnings.append(
        f"E1's full roster needs A1-A6; absent: {absent}. The leave-one-out grid can still run "
        "over the agents present, but the manuscript must report which roster was actually "
        "ablated rather than implying the full one.")

cm.write_json(NB13_DIR / "registry_manifest.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "13_agent_output_registry.ipynb",
    "seed": SEED,
    "n_registry_rows": int(len(registry)),
    "agents_present": sorted(present),
    "agents_absent": absent,
    "selected_arms_consensus_for_display": selected_arms,
    "selected_arms_per_fold": {a: {str(k): v for k, v in choices.items()}
                                for a, choices in per_fold_choice.items()},
    "selection_mode": "fold_specific_inner_validation",
    "selected_fold_columns": SELECTED_FOLD_COLUMNS,
    "inner_validation_images": len(inner_validation_keys),
    "intersection_images": len(intersection),
    "union_images": len(union),
    "full_cohort": len(truth_by_key),
    "reliability_scope": "inner_validation_only",
    "reliability_indexing": "per_fold",
    "inner_fold_columns": INNER_FOLD_COLUMNS if len(registry) else [],
    "leakage_note": (
        "Reliability is computed only on inner-validation rows, regenerated with Stage B's "
        "rule and seed. The gate asserts no fold's inner-validation rows are that fold's test "
        "images. This is what makes E3's metrics-aware arm legitimate rather than circular."),
})


def report(title, messages):
    print(title)
    for m in messages or []:
        print("  -", m)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
cm.write_json(NB13_DIR / "gate_nb13.json",
              {"passed": not failures, "failures": failures, "warnings": warnings})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 13 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 13 gate: PASSED")

## Notes carried forward

- **`agent_reliability_inner.csv` is the only reliability source E3 may use.** Its provenance is
  asserted in code, not by convention; that assertion is what distinguishes a metrics-aware
  result from a circular one, and the manuscript caption must say so.
- **Report the denominator.** If the all-agent intersection is smaller than the cohort, a fusion
  arm fitted on the intersection is not directly comparable to a single agent scored on
  everything. NB 14 and NB 15 both restate which set they used.
- **`agent_availability.csv` records absent agents.** E1's leave-one-out cannot remove an agent
  that was never there, and reporting a roster ablation over a partial roster as if it were the
  full one would misstate the result.
- One arm per agent enters the roster, chosen on inner-validation MAE. Choosing on test MAE
  would be selection on the outcome and would inflate every downstream comparison.